# Evo1 DNA Sequence Generation and Scoring

![Evo1 DNA Sequence Generation and Scoring](https://proto-bio.github.io/proto-assets/images/tool/evo1/hero.png)

This notebook demonstrates both tools that utilize Evo1 model. In the first section, we use `run_evo1_sample` to extend a short DNA prompt into a longer sequence using autoregressive sampling. In the second section, we use `run_evo1_score` to compute log-likelihood and perplexity metrics for a set of DNA sequences, which can be used to assess sequence quality or compare design candidates.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("evo1")
display_overview("evo1")
display_docs_section("evo1", "Background")

## Available tools

In [ ]:
display_available_tools("evo1")

### `run_evo1_sample`

Evo1 generates DNA sequences autoregressively from a starting prompt, extending a sequence nucleotide-by-nucleotide according to the model's learned distribution over prokaryotic and phage genomes. The `temperature` and `top_k` parameters control the diversity of the output: lower values produce more conservative, high-confidence extensions while higher values allow more exploratory generation. Fine-tuned checkpoints are also available for CRISPR and transposon-specific generation.

In [ ]:
from proto_tools import (
    Evo1SampleInput,
    Evo1SampleConfig,
    run_evo1_sample,
)

In [ ]:
# Dipslay docs
display_api_reference("evo1", "input", "run_evo1_sample")

# Seed generation with a real E. coli lacZ coding-sequence fragment.
inputs = Evo1SampleInput(prompts=["ATGACCATGATTACGGATTCACTGGCCGTCGTTTTACAACGTCGTGACTGG"])

In [ ]:
# Display config docs
display_api_reference("evo1", "config", "run_evo1_sample")

# Create a simple Evo1Sample config | see docs below for all fields
config = Evo1SampleConfig(
    model_name="evo-1-8k-base",
    max_new_tokens=200,
    temperature=0.7,
    top_k=4,
    prepend_prompt=True,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the sampling function
sample_result = run_evo1_sample(inputs, config)

In [ ]:
display_api_reference("evo1", "output", "run_evo1_sample")

# Select the prompt and corresponding generated output sequence and show some results
prompt = inputs.prompts[0]
sample = sample_result.results[0]
generated = sample.sequence
print(f"Prompt:       {prompt}")
print(f"Generated:    {generated[:80]}..." if len(generated) > 80 else f"Generated:    {generated}")
print(f"Total length: {len(generated)} nucleotides")
if sample.metrics:
    print(f"Avg log-likelihood: {sample.metrics['avg_log_likelihood']:.4f}")
    print(f"Perplexity:         {sample.metrics['perplexity']:.4f}")

### `run_evo1_score`

Evo1 scores sequences by computing the autoregressive log-likelihood at each position, measuring how well each nucleotide follows from its preceding context according to the model. The resulting perplexity is a single number summarizing sequence naturalness: lower perplexity indicates that the sequence more closely matches the statistical patterns of prokaryotic and phage genomes in the training data. This is useful for ranking generated candidates, filtering out low-quality designs, or comparing the naturalness of wild-type versus engineered sequences.

In [ ]:
from proto_tools import (
    Evo1ScoringInput,
    Evo1ScoringConfig,
    run_evo1_score,
)

In [ ]:
# Display docs
display_api_reference("evo1", "input", "run_evo1_score")

# Score two short DNA sequences
score_inputs = Evo1ScoringInput(sequences=["ATGCGTAAA", "ATGCGTAAATTT"])

In [ ]:
# Display config docs
display_api_reference("evo1", "config", "run_evo1_score")

# Create a simple Evo1Scoring config | see docs above for all fields
score_config = Evo1ScoringConfig(
    batch_size=2,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the scoring function
score_result = run_evo1_score(score_inputs, score_config)

In [ ]:
# Display output docs
display_api_reference("evo1", "output", "run_evo1_score")

# Show scoring results for each input sequence
for i, seq in enumerate(score_inputs.sequences):
    score = score_result.scores[i]
    print(f"Sequence {i + 1}: {seq}")
    print(f"    Log-likelihood:     {score.log_likelihood:.3f}")
    print(f"    Avg log-likelihood: {score.avg_log_likelihood:.3f}")
    print(f"    Perplexity:         {score.perplexity:.3f}")

## Export Results

Generated sequences and scoring metrics can be saved to standard file formats for use in downstream analysis pipelines. Sampling results support FASTA, TXT, and JSON export. Scoring results support CSV and JSON export.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

sample_result.export(name="evo1_sequences", export_path=output_dir, file_format="fasta")
print(f"Generated sequences exported to {output_dir / 'evo1_sequences.fasta'}")

score_result.export(name="evo1_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'evo1_scores.csv'}")